# 📗 그래프 알고리즘: 노드 유사도와 최단 경로

교안_01 에서는 노선망을 **커뮤니티**으로 나눴습니다. 이번에는 **공항과 공항 사이**를 봅니다.

- **어느 공항이 어느 공항과 닮았는가**: 이어진 상대 공항이 겹치는 정도로 "노선망에서 자리가 닮은 공항"을 찾습니다.
- **어디까지 어떻게 가는가**: 몇 편을 갈아타야 닿는지, 거리로 재면·시간으로 재면 길이 어떻게 달라지는지 잽니다.

마지막에는 커뮤니티·닮음·거리 셋을 겹쳐, **한 공항이 막혔을 때 대신 쓸 공항**을 명단으로 뽑습니다.

## 이 기술은 어디에 쓰일까요?

오늘 배우는 것은 전부 **그래프에서 순위를 매기고 길을 찾는 일**입니다. 노선망은 연습 재료일 뿐입니다.

| 현장의 질문 | 오늘의 도구 | 실제로 쓰이는 곳 |
|---|---|---|
| "이것과 **가장 닮은** 것은 무엇인가" | 노드 유사도 (1절) | 상품·동영상 추천, **중복 계정** 찾기, **대체 거래처** 고르기 |
| "A 에서 B 까지 **몇 단계**면 닿는가" | 최단 경로 (2절) | 지도 길찾기, 링크드인의 "2촌", 장애가 **번지는 경로** |
| "단계 수가 아니라 **거리나 시간**으로 재면" | 가중 최단 경로 (2-3) | 배송은 **시간**, 통신은 **지연**, 소개는 **신뢰** |
| "한쪽이 **막히면** 무엇으로 대신하나" | 커뮤니티·닮음·거리 겹치기 (3절) | 대체 공항, 대체 공급처, 대체 서버 고르기 |

**셋을 겹친다**는 것은 세 조건을 차례로 걸어 후보를 좁힌다는 뜻입니다. 대체 공항이라면 이렇습니다.

- **같은 커뮤니티**: 노선이 촘촘히 오가는 같은 권역이라 승객이 가려는 곳이 겹칩니다.
- **닮았다**: 이어진 상대 공항이 거의 같아 노선을 그대로 옮겨 받을 수 있습니다.
- **가깝다**: 두 편 안에 닿아 실제로 옮겨 타는 것이 가능합니다.

하나만 보면 엉뚱한 짝이 나옵니다. 권역만 맞추면 규모가 딴판인 공항이, 닮기만 따지면 멀리 떨어진 공항이 걸립니다.

**무엇이 끊기는지**는 이것을 뒤집은 질문입니다. 대신할 짝이 있는 자리는 막혀도 돌아가지만, 왕래가 **한 곳에 몰려 있는데** 그 자리를 대신할 짝이 없으면 거기 하나가 막히는 순간 연결이 통째로 끊깁니다. 3-2 에서 두 나라 사이 노선이 어디에 몰려 있는지 세어 보는 이유입니다.

이 과정 뒤쪽의 **GraphRAG 도 지식 그래프를 건너다니며** 답을 찾습니다. 그 건너기가 오늘의 **최단 경로**이고, "비슷한 개체까지 함께 찾아라"가 오늘의 **노드 유사도**입니다.

> 셋에 공통으로 붙는 **함정**: "가장 닮았다"·"가장 가깝다"는 **무엇을 세어 닮았다고 했는지·무엇을 비용으로 정했는지**에 따라 답이 통째로 바뀝니다.

## ⏪ 복습: 교안_01 까지

- 아시아 노선망을 **무방향으로 투영**하고 **Leiden** 으로 커뮤니티를 찾았습니다.
- 커뮤니티를 공항의 **국가·지역과 대조**했더니 NMI 가 0.754 안팎이었습니다. 권역이 통째로 한 커뮤니티가 되기도, 한 나라가 갈라지기도 합니다.
- 어긋난 자리는 오류가 아니라 **국경과 상관없이 촘촘한 노선 덩어리**였습니다. 오늘은 그 덩어리를 **공항 단위로** 파고듭니다.
- 재현하려면 `randomSeed` 와 `concurrency: 1` 을 **함께** 줘야 했습니다. 3절에서 그대로 씁니다.

**오늘의 목표**

**1. 노드 유사도**
- [ ] (1-1) `gds.nodeSimilarity` 의 `topK`·`degreeCutoff` 를 쓰고, **상대가 적은 공항이 유사도 1.0 을 독차지하는 함정**을 피한다.
- [ ] (1-2) **Jaccard 유사도**가 무엇을 무엇으로 나눈 값인지 알고 손으로 검산한다.

**2. 최단 경로**
- [ ] (2-1) `gds.shortestPath.dijkstra` 로 **몇 편을 갈아타는지** 잰다.
- [ ] (2-2) `gds.allShortestPaths.dijkstra` 로 한 공항에서 전체까지의 거리 분포를 본다.
- [ ] (2-3) **편 수 최소·거리 최소·시간 최소**가 서로 다른 길을 고르는 것을 한 표로 확인한다.

**3. 셋을 겹쳐 대체 공항 찾기**
- [ ] (3-1) 커뮤니티·유사도·경로를 겹쳐 **대체 공항 후보**를 명단으로 뽑아낸다.

아래 준비 셀을 **위에서부터** 실행하세요. 연결 → 초기화 → 투영 정리 → 세 그래프 적재·투영 순서입니다.

그래프는 **세 개**입니다. 시연은 항공 노선망, 따라하기는 1절·3절이 메일망, 2절이 전철망입니다. 레이블이 `:Airport`·`:Member`·`:Station` 으로 달라 **한 데이터베이스에 함께 둬도 섞이지 않습니다.** 투영 이름도 `air`·`team`·`subway` 로 갈라 둡니다.

In [ ]:
# [제공 코드] Neo4j 연결: 교재 실습은 로컬 Neo4j (bolt://localhost:7689) 전용입니다.
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: 교재는 100% 로컬 Neo4j 인스턴스 (7689) 강제
env_file = os.path.join(os.path.abspath(''), '.env')
if os.path.exists(env_file):
    load_dotenv(env_file, override=True)

# 상위 클라우드 Aura 환경변수가 상속되었을 경우 로컬 7689로 강제 치환
raw_uri = os.getenv('NEO4J_URI', '')
if not raw_uri or 'databases.neo4j.io' in raw_uri:
    NEO4J_URI = 'bolt://localhost:7689'
    NEO4J_USER = 'neo4j'
    NEO4J_PASSWORD = 'test0011'
else:
    NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
    NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
    NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'test0011')

# 2) 드라이버 연결
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

# 3) 공용 Cypher 실행 헬퍼
def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print('Neo4j 연결:', NEO4J_URI)


In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Airport', 'Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'Member', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'Person', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 남아 있는 GDS 투영(메모리 그래프)을 모두 정리: 이 셀은 실행만 하세요.
# 앞 실습의 사본이 남아 있으면 같은 이름으로 다시 투영할 때 충돌합니다.
for row in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($g) YIELD graphName", g=row["graphName"])
print("남은 투영:", run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"))

<img src="images/교안/항공_아시아망_구성.png" width="900">

**시연에 쓰는 그래프**입니다.

| 레이블 | 뜻 | 속성 |
|---|---|---|
| `:Airport` | 공항 767곳 | `iata` 공항 코드 · `name`·`city` 이름과 도시 · `country` 국가·지역 47종 · `lat`·`lon` 좌표 |

| 관계 | 잇는 것 | 속성 |
|---|---|---|
| `ROUTE` | 공항 → 공항 (방향별 8,045건) | `km` 두 공항 사이 거리 · `hours` km/800 + 1 · `airlines` 운항 항공사 수 |

> **`km`·`hours` 는 원본에 없는 값입니다.** `km` 은 두 공항 좌표로 계산했고, `hours` 는 준비 셀이 `km/800 + 1` 로 새깁니다. 2-3 에서 이 정의가 답을 어떻게 바꾸는지 봅니다. 출처는 마무리 절에 있습니다.

In [ ]:
# [제공 코드] 아시아 항공 노선망 적재: 이 셀은 실행만 하세요(실습에 쓸 그래프를 만듭니다).
import pandas as pd

# 공항 767행, 노선 8,045행(방향별)을 CSV 에서 읽습니다.
airports = pd.read_csv("data/airports_asia_nodes.csv")   # iata, name, city, country, lat, lon
routes = pd.read_csv("data/airports_asia_edges.csv")     # from_iata, to_iata, km, hours, airlines

# iata 로 공항을 찾을 때 전체를 훑지 않도록 유일성 제약을 먼저 겁니다(인덱스가 함께 생깁니다).
run_cypher("CREATE CONSTRAINT airport_iata IF NOT EXISTS FOR (a:Airport) REQUIRE a.iata IS UNIQUE")

# 공항을 노드로 만듭니다.
run_cypher("""UNWIND $rows AS r
    CREATE (:Airport {iata: r.iata, name: r.name, city: r.city, country: r.country,
                      lat: r.lat, lon: r.lon})""", rows=airports.to_dict("records"))

# 노선을 관계로 잇습니다. 왕복이면 CSV 에 두 행이 있어 관계도 둘이 됩니다.
run_cypher("""UNWIND $rows AS r
    MATCH (a:Airport {iata: r.from_iata}), (b:Airport {iata: r.to_iata})
    CREATE (a)-[:ROUTE {km: r.km, hours: r.hours, airlines: r.airlines}]->(b)""",
           rows=routes.to_dict("records"))

print("공항:", run_cypher("MATCH (a:Airport) RETURN count(a) AS n")[0]["n"],
      "· 노선 관계(방향별):", run_cypher("MATCH (:Airport)-[r:ROUTE]->() RETURN count(r) AS n")[0]["n"])

In [ ]:
# [제공 코드] 관계에 hours(소요 시간)를 정의합니다: 이 셀은 실행만 하세요.
# 원본에 '시간'이 없어 우리가 정합니다. 이 상수를 바꾸면 뒤의 최단 시간 경로가 달라집니다.
run_cypher("""MATCH ()-[r:ROUTE]->() SET r.hours = round(r.km / 800.0 + 1.0, 2)""")
print(run_cypher("MATCH ()-[r:ROUTE]->() RETURN min(r.hours) AS shortest, max(r.hours) AS longest, count(*) AS routes"))

In [ ]:
# [제공 코드] 아시아 항공 노선망을 무방향으로 투영합니다: 이 셀은 실행만 하세요.
# 기존 air 투영이 있다면 제거합니다.
run_cypher("CALL gds.graph.drop('air', false) YIELD graphName")   # false: 없으면 그냥 넘어간다

# 무방향으로 투영하면서 2-3 에서 쓸 관계 속성도 함께 싣습니다(투영은 그 순간의 사본이라 나중에 실을 수 없습니다).
stats = run_cypher('''
    CALL gds.graph.project('air', 'Airport',
      {ROUTE: {orientation: 'UNDIRECTED', properties: ['km', 'hours', 'airlines']}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

<img src="images/교안/메일_다섯부서_구성.png" width="900">

**1절·3절 따라하기에 쓰는 그래프**입니다.

| 레이블 | 뜻 | 속성 |
|---|---|---|
| `:Member` | 구성원 194명 | `employee_id` 익명 사번 · `dept` 부서 5종 |

| 관계 | 잇는 것 | 속성 |
|---|---|---|
| `MAILED` | 구성원 → 구성원 (방향별 2,439건) | 없음 |

In [ ]:
# [제공 코드] 다섯 부서 부분망 적재: 이 셀은 실행만 하세요(실습에 쓸 그래프를 만듭니다).
import pandas as pd

# 항공망의 :Airport 와 레이블도 관계 이름도 달라서 서로 섞이지 않습니다.
people = pd.read_csv("data/email_dept_group2_nodes.csv")   # employee_id, dept
mails = pd.read_csv("data/email_dept_group2_edges.csv")     # from_id, to_id

# employee_id 로 사람을 찾을 때 전원을 훑지 않도록 유일성 제약을 먼저 겁니다.
run_cypher("CREATE CONSTRAINT member_id IF NOT EXISTS FOR (p:Member) REQUIRE p.employee_id IS UNIQUE")

# 구성원을 노드로 만듭니다.
run_cypher("UNWIND $rows AS r CREATE (:Member {employee_id: r.employee_id, dept: r.dept})",
           rows=people.to_dict("records"))

# 메일을 관계로 잇습니다. 서로 주고받은 사이면 관계가 둘입니다.
run_cypher("""UNWIND $rows AS r
    MATCH (a:Member {employee_id: r.from_id}), (b:Member {employee_id: r.to_id})
    CREATE (a)-[:MAILED]->(b)""", rows=mails.to_dict("records"))

print("구성원:", run_cypher("MATCH (p:Member) RETURN count(p) AS n")[0]["n"],
      "· 메일 관계:", run_cypher("MATCH (:Member)-[r:MAILED]->() RETURN count(r) AS n")[0]["n"])

In [ ]:
# [제공 코드] 다섯 부서 부분망을 무방향으로 투영합니다: 이 셀은 실행만 하세요.
# 기존 team 투영이 있다면 제거합니다.
run_cypher("CALL gds.graph.drop('team', false) YIELD graphName")

# 무방향으로 투영합니다. 메일 관계에는 거리도 시간도 없어 관계만 싣습니다
# (사람과 사람을 몇 다리 건너 닿는지, 곧 관계 수로만 잽니다).
stats = run_cypher('''
    CALL gds.graph.project('team', 'Member',
      {MAILED: {orientation: 'UNDIRECTED'}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

<img src="images/교안/전철_구성.png" width="900">

**2절 따라하기에 쓰는 그래프**입니다. 30일차에서 쓴 것과 같은 파일입니다.

| 레이블 | 뜻 | 속성 |
|---|---|---|
| `:Station` | 역 659개 | `name` 역 이름 · `lines` 그 역을 지나는 노선 목록 |

| 관계 | 잇는 것 | 속성 |
|---|---|---|
| `NEXT_TO` | 역 → 이웃 역 (778건) | `line` 이 구간의 노선 · `km` 두 역 좌표 사이 직선거리 |

> **`km` 은 원본에 없는 값입니다.** 두 역 좌표 사이 직선거리로 계산했습니다. 구간은 한 방향으로만 저장하고 조회는 방향 없이 합니다.

In [ ]:
# [제공 코드] 수도권 전철망 적재: 이 셀은 실행만 하세요(30일차에서 쓴 것과 같은 파일입니다).
# 스키마: (:Station {name, lines})-[:NEXT_TO {line, km}]->(:Station)
import pandas as pd

stations = pd.read_csv("data/seoul_subway_stations.csv")   # name, lines, lat, lon
sections = pd.read_csv("data/seoul_subway_edges.csv")      # from, to, line, km

# 이름으로 찾을 일이 많으니 인덱스부터 만듭니다(적재 속도가 여기서 갈립니다).
run_cypher("CREATE INDEX station_name IF NOT EXISTS FOR (s:Station) ON (s.name)")

# 역을 노드로 만듭니다. lines 는 split 으로 목록에 담습니다(문자열이면 '1호선' 이 '인천1호선' 까지 걸립니다).
run_cypher("UNWIND $rows AS r CREATE (:Station {name: r.name, lines: split(r.lines, '|')})",
           rows=stations[["name", "lines"]].to_dict("records"))

# 구간을 관계로 잇습니다. 저장은 한 방향뿐이고 조회는 방향 없이 합니다.
run_cypher("""UNWIND $rows AS r
    MATCH (a:Station {name: r.from_name}), (b:Station {name: r.to_name})
    CREATE (a)-[:NEXT_TO {line: r.line, km: r.km}]->(b)""",
           rows=sections.rename(columns={"from": "from_name", "to": "to_name"}).to_dict("records"))

print("역:", run_cypher("MATCH (s:Station) RETURN count(s) AS n")[0]["n"],
      "· 구간:", run_cypher("MATCH (:Station)-[r:NEXT_TO]->() RETURN count(r) AS n")[0]["n"])

In [ ]:
# [제공 코드] 수도권 전철망을 무방향으로 투영합니다: 이 셀은 실행만 하세요.
# 기존 subway 투영이 있다면 제거합니다.
run_cypher("CALL gds.graph.drop('subway', false) YIELD graphName")

# 열차는 양쪽으로 다니므로 무방향으로 투영하고, 2절에서 쓸 km 도 함께 싣습니다.
stats = run_cypher('''
    CALL gds.graph.project('subway', 'Station',
      {NEXT_TO: {orientation: 'UNDIRECTED', properties: ['km']}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

---
# 1. 노드 유사도: 어느 공항과 노선망의 자리가 닮았나

교안_01 에서 찾은 커뮤니티는 **덩어리** 단위였지만, 실제 질문은 대개 하나씩입니다. 먼저 유사도를 구해 함정을 걷어 내고, 값이 맞는지 손으로 검산합니다.

## 1-1. 이어진 상대가 얼마나 겹치는가

### 왜 필요할까요?
"이 공항이 폐쇄되면 어디로 돌릴 수 있나". 이런 질문에는 **이어진 상대 공항이 얼마나 겹치는가**가 답이 됩니다.

### 문법: Jaccard 유사도
두 공항의 **상대 공항 집합**을 견줍니다. 한쪽의 상대 공항 집합을 $A$, 다른 쪽을 $B$ 라 하면 이렇습니다.

$$ \text{Jaccard}(A,\, B) \;=\; \frac{|A \cap B|}{|A \cup B|} \;=\; \frac{\text{둘 다와 이어진 공항 수}}{\text{둘 중 하나라도와 이어진 공항 수}} $$

$|A \cap B|$ 는 교집합의 크기, $|A \cup B|$ 는 합집합의 크기입니다.

<img src="images/교안/유사도_계산_세단계.png" width="860">

인천과 나리타로 세어 본 그림입니다. 합집합은 두 집합의 크기를 더한 뒤 **겹친 만큼 한 번 빼야** 합니다. 그러지 않으면 겹친 51곳을 두 번 세게 됩니다. 이 계산은 1-2 에서 Cypher 로 직접 해 봅니다.

0 이면 겹치는 상대가 없고 1 이면 상대 목록이 같습니다. **관계가 아니라 관계의 모양**을 비교하므로, 두 공항 사이에 노선이 없어도 유사도가 높을 수 있습니다.

```cypher
CALL gds.nodeSimilarity.stream('투영이름', { topK: 10 })
YIELD node1, node2, similarity
```
- `topK` 는 **공항마다 상위 몇 곳까지** 돌려줄지입니다. 기본값 10.
- 서로가 서로의 상위에 들면 (A, B) 와 (B, A) 가 **둘 다** 나옵니다. 한쪽만 나올 수도 있습니다.
- 기본 지표는 Jaccard 이고 `similarityMetric: 'OVERLAP'` 은 **작은 쪽 집합으로 나눕니다.** 상대 수가 크게 차이 나는 쌍에서 답이 달라집니다.

In [ ]:
import pandas as pd

# 이웃이 겹치는 정도로 공항 쌍의 유사도를 재고, 높은 순으로 여섯 쌍만 본다
# topK: 10 은 '공항마다 상위 열 곳'이다. 전체에서 열 쌍만 고르는 것이 아니다
sim_rows = run_cypher('''
    CALL gds.nodeSimilarity.stream('air', { topK: 10, degreeCutoff: 1 })
    YIELD node1, node2, similarity
    RETURN gds.util.asNode(node1).iata AS airport,
           gds.util.asNode(node2).iata AS partner,
           round(similarity, 4) AS similarity
    ORDER BY similarity DESC, airport LIMIT 6''')
display(pd.DataFrame(sim_rows))

> **유사도 1.0 이 줄줄이 나옵니다.** 함정입니다. 몇 쌍이나 되는지 세고, 그중 한 쌍의 **상대가 몇 곳인지** 보면 정체가 드러납니다.

In [ ]:
# 같은 쌍이 (A,B), (B,A) 두 줄로 나올 수 있어 코드 순서를 맞춰 한 쌍으로 센다
ones = run_cypher('''
    CALL gds.nodeSimilarity.stream('air', { topK: 10, degreeCutoff: 1 })
    YIELD node1, node2, similarity
    WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
    WHERE similarity >= 1.0
    RETURN count(DISTINCT CASE WHEN a.iata < b.iata THEN [a.iata, b.iata]
                               ELSE [b.iata, a.iata] END) AS pairs''')
print('유사도가 1.0 인 공항 쌍:', ones[0]['pairs'], '쌍')

In [ ]:
# 그중 한 쌍을 골라 상대가 몇 곳인지 센다(관계 수가 아니라 서로 다른 공항 수)
low = run_cypher('''
    MATCH (a:Airport { iata: $a })--(x:Airport)
    WITH a, count(DISTINCT x) AS partners_a   // 왕복 노선은 관계가 둘이라 DISTINCT 로 한 번만 센다
    MATCH (b:Airport { iata: $b })--(y:Airport)
    RETURN a.iata AS a, a.city AS city_a, partners_a,
           b.iata AS b, b.city AS city_b, count(DISTINCT y) AS partners_b''',
    a='AAT', b='HTN')
print(low[0])   # partners_a 와 partners_b 는 각 공항의 상대 공항 수. 둘 다 아주 적어 유사도가 쉽게 1.0 이 된다

> 1.0 인 쌍이 **478쌍**이나 됩니다. 그중 한 쌍인 알타이(AAT)와 허톈(HTN)은 각각 상대가 **1곳뿐**입니다. 상대가 한 곳이면 그 한 곳을 공유하는 순간 교집합도 1, 합집합도 1 이라 유사도가 자동으로 1.0 입니다. **닮은 게 아니라 비교할 것이 없는 것입니다.**

막는 장치가 `degreeCutoff` 입니다. **서로 다른 상대 공항이 그 수보다 적은 공항은 비교에서 아예 빼 버립니다.** 세는 것은 관계 수가 아니라 상대 공항 수입니다. 왕복 노선은 관계가 둘이어도 상대는 한 곳으로 셉니다.

In [ ]:
# degreeCutoff 를 올려 가며 비교 대상 수와 최고 쌍이 어떻게 바뀌는지 본다
for cutoff in [1, 10, 20]:
    stat = run_cypher('''
        CALL gds.nodeSimilarity.stats('air', { topK: 10, degreeCutoff: $cutoff })
        YIELD nodesCompared RETURN nodesCompared''', cutoff=cutoff)[0]
    # 같은 조건으로 stats(요약)와 stream(개별 쌍)을 각각 불러 본다
    top = run_cypher('''
        CALL gds.nodeSimilarity.stream('air', { topK: 10, degreeCutoff: $cutoff })
        YIELD node1, node2, similarity
        RETURN gds.util.asNode(node1).iata AS airport,
               gds.util.asNode(node2).iata AS partner,
               round(similarity, 4) AS similarity
        ORDER BY similarity DESC, airport LIMIT 1''', cutoff=cutoff)[0]
    print(f"cutoff={cutoff:2d} · 비교 대상 {stat['nodesCompared']:4d}곳 · 최고 쌍 {top}")

> `degreeCutoff` 를 1 에서 10, 20 으로 올리면 비교 대상이 767곳에서 187곳, 112곳으로 줄어듭니다. cutoff 1 의 최고 쌍은 방금 본 1.0 쌍 가운데 하나라 아무 뜻이 없습니다. 10 으로 올리면 최고 쌍이 **하노이(HAN)와 호찌민(SGN)(유사도 0.6604)** 로 바뀝니다. 상대가 각각 42곳·46곳인데 합친 목록의 3분의 2가 공통입니다. 한 나라 안의 두 관문 공항이라 노선망에서 서 있는 자리가 거의 같습니다.

**얼마로 잡아야 할까요?** 낮으면 쓰레기 쌍이 상위를 채우고, 높으면 작은 공항을 놓칩니다. `degreeCutoff` 가 자르는 기준은 **서로 다른 상대 공항 수**이고, 이 노선망에서 그 중앙값은 **4곳**입니다. 반면 **차수**(관계 수)의 중앙값은 **6** 입니다. 무방향 투영이 왕복 노선을 두 관계로 세기 때문입니다. **둘을 헷갈리면 기준을 두 배로 잘못 잡습니다.**

**10 은 센 기준입니다.** 상대가 열 곳이 안 되는 공항이 전체의 76% 라, 767곳 가운데 187곳만 비교에 남습니다. 몇으로 잡을지는 데이터마다 다르니 **상대 공항 수의 분포를 먼저 보고 정합니다.** 이 교안은 "관문급 공항끼리만 견주자"는 뜻으로 10 을 씁니다.

In [ ]:
# 인천은 상대가 많아 cutoff 를 어디에 두든 이 목록은 그대로다(작은 공항은 상위에 못 든다)
near = run_cypher('''
    CALL gds.nodeSimilarity.stream('air', { topK: 5, degreeCutoff: 10 })
    YIELD node1, node2, similarity
    WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
    // 왼쪽이 인천인 행만 남긴다. 그러면 오른쪽 b 가 '닮은 공항'이 된다
    WHERE a.iata = $who
    RETURN b.iata AS airport, b.city AS city, b.country AS country,
           round(similarity, 4) AS similarity
    ORDER BY similarity DESC''',
    who='ICN')
display(pd.DataFrame(near))

> 인천(ICN)과 가장 닮은 공항은 **나리타(NRT) 0.5258**, 그다음이 **타이베이(TPE) 0.5254** 입니다. 노선을 주고받는 상대가 거의 같은 동아시아 관문(나라 밖 노선이 몰리는 공항) 공항들이 위에 모입니다.

**소수 넷째 자리까지 적은 이유.** 1위와 2위의 차이가 0.0004 밖에 안 됩니다(반올림 전 차이는 0.0003). 둘째 자리로 반올림하면 둘이 같은 값이 되어 **순위가 사라집니다.** 자리를 줄이기 전에 1위와 2위의 간격을 확인하십시오.

## 1-2. 손으로 검산하기
유사도 값을 그냥 믿지 말고 한 쌍만 직접 세어 봅시다. 정의대로면 `교집합 / 합집합` 이어야 합니다.

In [ ]:
# Jaccard 를 순수 Cypher 로 손 검산한다
check = run_cypher('''
    MATCH (a:Airport { iata: $a })--(x:Airport)
    WITH collect(DISTINCT x) AS partners_a   // 왕복 노선은 관계가 둘이라 DISTINCT 로 한 번만 담는다
    MATCH (b:Airport { iata: $b })--(y:Airport)
    WITH partners_a, collect(DISTINCT y) AS partners_b
    WITH partners_a, partners_b,
         [p IN partners_a WHERE p IN partners_b] AS both
    RETURN size(partners_a) AS partners_a, size(partners_b) AS partners_b,
           size(both) AS common,
           size(partners_a) + size(partners_b) - size(both) AS union_size,   // 겹친 만큼 뺀다
           round(1.0 * size(both) / (size(partners_a) + size(partners_b) - size(both)),
                 4) AS jaccard''',
    a='ICN', b='NRT')
print(check[0])   # common 은 둘 다의 상대 수, union_size 는 합집합 크기, jaccard 는 common 을 union_size 로 나눈 값

In [ ]:
# 관계 수를 세어 상대 공항 수와 견준다(왕복 노선은 둘로 세어진다. 그게 곧 차수다)
degree = run_cypher('''
    MATCH (a:Airport { iata: $a })-[r:ROUTE]-()
    RETURN count(r) AS rels''',
    a='ICN')[0]['rels']
print('관계 수(방향별):', degree)

> 인천(ICN)의 상대는 94곳, 나리타(NRT)의 상대는 54곳, 둘 다와 이어진 공항이 **51곳**, 합집합이 **97곳**입니다. 51 나누기 97 는 **0.5258**. GDS 가 돌려준 값과 같습니다. **한 쌍만 손으로 검산**해 두면 나중에 값이 이상할 때 지표와 데이터 중 무엇을 의심할지 가릅니다.

인천(ICN)의 **관계 수는 188개**인데 **상대 공항은 94곳**입니다. 왕복 노선이 관계 둘로 들어 있어 딱 두 배입니다.

<img src="images/교안/유사도_정의와_함정.png" width="860">

**왼쪽이 진짜 닮은 짝, 오른쪽이 함정**이고, 오른쪽 아래 상자가 그 함정을 `degreeCutoff` 로 걷어 낸다는 뜻입니다. 차이는 값이 아니라 **분모**입니다. 합집합이 97곳일 때의 유사도와 1곳일 때의 1.0 은 같은 숫자가 아닙니다.

> 3절의 복선입니다. `degreeCutoff` 로 작은 공항을 걷어 낸 뒤, 국가·지역이 **다른** 공항 쌍 가운데 가장 닮은 것은 쿠알라룸푸르(KUL)와 싱가포르(SIN), 유사도 0.5868 입니다. 3절에서 이런 쌍만 골라내 **대체 공항 명단**을 만듭니다.

### 🖐️ 함께 따라하기: 다섯 부서 메일망에서 닮은 사람 찾기

그래프를 바꿔 같은 도구를 써 봅니다. `'team'` 투영에서 **377번**과 가장 닮은 사람 **다섯 명**을 유사도가 높은 순서로 출력하세요. `degreeCutoff: 10`, `topK: 5` 를 주고, 상대의 **부서 번호도 함께** 보여 주세요.

**확인 기준**: 1위는 **157번**(부서 0, 유사도 **0.68**)입니다. 377번은 부서 7 인데 **1위가 다른 부서 사람**입니다. 조직도가 갈라 놓은 두 사람이 실제로는 같은 사람들을 상대합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.nodeSimilarity.stream 을 'team' 투영에 부른다(topK 와 degreeCutoff 는 지문의 값)
# 2) node1, node2 를 gds.util.asNode 로 노드로 되돌린 뒤
#    왼쪽이 377번인 행만 남긴다
# 3) b.employee_id, b.dept, round(similarity, 4) 를 유사도 내림차순으로 돌려받아 출력한다

### ✅ 바로 확인 퀴즈

**1.** Jaccard 유사도는 무엇을 무엇으로 나눈 값인가요?

<details><summary>정답 보기</summary>

**교집합(둘 다와 이어진 상대 수)을 합집합(둘 중 하나라도와 이어진 상대 수)으로** 나눈 값. 0 에서 1 사이입니다.

</details>

**2.** 상위 유사도가 1.0 인 쌍만 잔뜩 나왔습니다. 무엇을 먼저 확인해야 할까요?

<details><summary>정답 보기</summary>

**그 공항들의 상대 수**입니다. 상대가 한두 곳뿐이면 유사도가 쉽게 1.0 이 됩니다. `degreeCutoff` 로 걸러 냅니다.

</details>

**3.** 결과에 (A, B) 와 (B, A) 가 둘 다 나왔습니다. 버그일까요?

<details><summary>정답 보기</summary>

아닙니다. `topK` 는 **공항마다** 상위 K 곳을 돌려주니 서로가 서로의 상위에 들면 두 줄이 됩니다. 한 줄만 보려면 `WHERE a.iata < b.iata` 로 거르되, **한쪽에만 나온 쌍이 통째로 사라질 수 있습니다.**

</details>

---
# 2. 최단 경로: 어떻게 가야 가장 빠른가

"인천에서 저기까지 직항이 없는데, 몇 편을 갈아타야 하나." 그래프에서는 **최단 경로**로 답합니다. 물류의 배송 경로, 지하철 환승, 추천의 연결 고리가 모두 같은 문제입니다.

## 2-1. 몇 편을 갈아타야 닿는가

### 왜 필요할까요?
출발지와 목적지를 콕 집어 "몇 편인가"를 잽니다. 이 수가 뒤의 분포·비용 이야기의 기준입니다.

### 문법: `gds.shortestPath.dijkstra`
```cypher
MATCH (a:Airport { iata: 'ICN' }), (b:Airport { iata: 'BKK' })
CALL gds.shortestPath.dijkstra.stream('투영이름', { sourceNode: a, targetNode: b })
YIELD totalCost, nodeIds
```
- **노드 자체**(`a`, `b`)를 넘깁니다. `iata` 같은 우리 데이터의 코드는 안 됩니다.
- `totalCost` 는 총비용, `nodeIds` 는 지나간 노드 목록입니다.
- **가중치를 안 주면 관계 하나가 비용 1** 이라, `totalCost` 가 곧 **편 수**가 됩니다.

> 무방향 투영이라 **원본에 한 방향만 있는 95쌍도 양쪽으로 오갈 수 있는 것처럼** 계산됩니다. 표를 끊는 문제였다면 방향을 살려야 합니다. 아래에서 재는 구간은 모두 왕복이라 답은 바뀌지 않습니다.

In [ ]:
# 출발 공항에서 도착 공항까지 갈아타는 횟수가 가장 적은 길 하나를 찾는다
# 가중치를 주지 않았으므로 관계 하나가 비용 1 이고, totalCost 와 편 수가 같은 값이 된다
route = run_cypher('''
    MATCH (a:Airport { iata: $a }), (b:Airport { iata: $b })
    CALL gds.shortestPath.dijkstra.stream('air', { sourceNode: a, targetNode: b })
    YIELD totalCost, nodeIds
    RETURN totalCost AS cost, size(nodeIds) - 1 AS hops,
           [n IN nodeIds | gds.util.asNode(n).iata] AS route,
           [n IN nodeIds | gds.util.asNode(n).city] AS cities''',
    a='ICN', b='BTH')
print('총비용:', route[0]['cost'], '· 편 수:', route[0]['hops'])

In [ ]:
print('경로:', route[0]['route'])
print('지나는 도시:', route[0]['cities'])

In [ ]:
# 2편으로 가는 길이 몇 개인지 센다(최단이 2편일 때만 쓰는 패턴이다. 갈아타는 공항 한 곳이 곧 길 하나다)
vias = run_cypher('''
    MATCH (a:Airport { iata: $a })-[:ROUTE]-(v:Airport)-[:ROUTE]-(b:Airport { iata: $b })
    RETURN count(DISTINCT v) AS paths,   // DISTINCT 가 없으면 같은 경유지를 왕복 노선 수만큼 센다
           collect(DISTINCT v.iata + ' ' + v.city) AS vias''',
    a='ICN', b='BTH')
print(vias[0])   # paths 는 2편으로 가는 길의 가짓수, vias 는 갈아타는 공항 목록

> 바탐(BTH)은 인도네시아의 섬 도시입니다. 인천(ICN)에서 직항이 없어 **2편**을 갈아타야 합니다.

그런데 2편짜리 길이 **2개**나 됩니다. 위 셀이 돌려준 길은 발리 덴파사르(DPS) 경유였습니다. **가중치 없는 최단 경로는 동점이 여럿이라 어느 길이 나오는지는 보장이 없습니다.**

그러니 근거로 쓸 수 있는 것은 **편 수 2** 뿐입니다. 경로를 보고서에 적으려면 "이 실행에서는 이렇게 나왔다"라고 밝히거나, 2-3 처럼 **동점이 생기지 않는 잣대**로 다시 재야 합니다.

## 2-2. 한 공항에서 전체까지: `gds.allShortestPaths.dijkstra`
두 공항만 보면 그 값이 큰지 작은지 알 수 없습니다. **한 공항에서 전체까지의 거리 분포**를 보면 노선망이 얼마나 좁은지 한눈에 들어옵니다.

In [ ]:
# 이 공항에서 1편·2편·3편에 닿는 공항이 각각 몇 곳인지 센다
# targetNode 를 주지 않으면 도착지를 정하지 않고 닿는 공항 전부까지의 거리를 돌려준다
spread = run_cypher('''
    MATCH (a:Airport { iata: $a })
    CALL gds.allShortestPaths.dijkstra.stream('air', { sourceNode: a })
    YIELD totalCost
    RETURN toInteger(totalCost) AS hops, count(*) AS airports
    ORDER BY hops''', a='ICN')
display(pd.DataFrame(spread))

In [ ]:
# 이번에는 상대 공항이 가장 많은 허브를 기준으로 같은 것을 잰다
# PEK 는 이 노선망에서 상대 공항이 가장 많다(157곳)
hub_spread = run_cypher('''
    MATCH (a:Airport { iata: $a })
    CALL gds.allShortestPaths.dijkstra.stream('air', { sourceNode: a })
    YIELD totalCost
    RETURN toInteger(totalCost) AS hops, count(*) AS airports
    ORDER BY hops''', a='PEK')
display(pd.DataFrame(hub_spread))

> **직항 수는 크게 다릅니다.** 인천(ICN)은 94곳, 베이징(PEK)은 157곳입니다. 1.7배 차이입니다. **그런데 두 편 안에 닿는 공항은 605곳 대 597곳으로 거의 같고, 가장 먼 곳도 둘 다 4편입니다.**

**허브의 이점은 '닿느냐'가 아니라 '직항이 있느냐'에서 나옵니다.** 한 번 갈아탈 각오를 하면 어디서 출발하든 노선망의 대부분에 닿습니다.

> 표의 첫 줄(0편, 1곳)은 출발지 자신입니다. 오늘 쓰는 데이터는 **가장 큰 연결 덩어리만** 남긴 것이라, 어디서 출발해도 모든 공항에 닿습니다.

## 2-3. 무엇을 최소화할지 정하면 다른 길이 나온다

여기까지는 관계 하나를 비용 1 로 봤습니다. 하지만 표를 끊을 때 줄이고 싶은 것은 대개 **거리**나 **시간**입니다.

관계에는 `km`(대권 거리)과 `hours`(소요 시간)가 실려 있습니다. 이 값을 `relationshipWeightProperty` 로 넘기면 Dijkstra 가 **그 값의 합이 가장 적은 길**을 찾습니다.

> **같은 인자를 읽는 방향이 반대입니다.** 교안_01 의 커뮤니티 탐지는 `relationshipWeightProperty` 를 **무게**로 읽어 `airlines` 가 클수록 가까운 사이였습니다. Dijkstra 는 **거리**로 읽어 클수록 멉니다. 그래서 속성 이름에 `km`·`hours` 처럼 **단위를 그대로** 붙였습니다. **이름이 읽는 방향을 알려 주게** 짓는 것이 실수를 막는 가장 싼 방법입니다.

In [ ]:
# 잣대 셋으로 같은 구간을 재서 나란히 본다. 가중치를 주지 않으면 편 수 최소가 된다
compare = []
for label, weight in [('편 수 최소', None), ('거리 최소', 'km'), ('시간 최소', 'hours')]:
    weight_line = "" if weight is None else f", relationshipWeightProperty: '{weight}'"
    # nodeIds 에는 출발지도 들어 있어서 1 을 빼야 편 수가 된다
    rows = run_cypher(f'''
        MATCH (a:Airport {{ iata: $a }}), (b:Airport {{ iata: $b }})
        CALL gds.shortestPath.dijkstra.stream('air',
             {{ sourceNode: a, targetNode: b{weight_line} }})
        YIELD totalCost, nodeIds
        RETURN size(nodeIds) - 1 AS hops,
               [n IN nodeIds | gds.util.asNode(n).iata] AS route''',
        a='ICN', b='BTH')
    route = rows[0]['route']
    # 세 길을 견주려면 각 길의 거리와 시간을 모두 재야 한다. 구간마다 min 으로 왕복 중복을 없앤다
    totals = run_cypher('''
        // range 로 0,1,2… 를 만들어 경로 목록의 이웃한 두 칸을 짝지어 구간을 하나씩 훑는다
        UNWIND range(0, size($ids) - 2) AS step
        MATCH (x:Airport { iata: $ids[step] })-[r:ROUTE]-(y:Airport { iata: $ids[step + 1] })
        WITH step, min(r.km) AS km, min(r.hours) AS hours
        RETURN sum(km) AS km, round(sum(hours), 2) AS hours''', ids=route)[0]
    compare.append({'기준': label, '편 수': rows[0]['hops'], 'km': totals['km'],
                    '시간': totals['hours'], '경로': ' → '.join(route)})
display(pd.DataFrame(compare))

<img src="images/교안/홉_거리_시간_세잣대.png" width="860">

> 인천(ICN)에서 바탐(BTH)까지, 세 줄이 전부 다른 길입니다.

| 잣대 | 편 수 | 거리 | 시간 | 경유 |
|---|---|---|---|---|
| 편 수 최소 | 2편 | 6,916km | 10.65시간 | 발리 덴파사르(DPS) (동점 2개 중 하나) |
| 거리 최소 | 4편 | 5,096km | 10.37시간 | 광저우(CAN)·페낭(PEN)·수방(SZB) |
| 시간 최소 | 3편 | 5,175km | 9.47시간 | 쿠알라룸푸르(KUL)·페칸바루(PKU) |

**한 번 더 갈아타는데 시간은 덜 걸립니다.** 시간 최소 길은 3편이라 한 번 더 갈아타지만 9.47시간으로 10.65시간보다 짧습니다. 편 수 최소 길이 크게 돌아가기 때문입니다(6,916km 대 5,175km).

**거리 최소는 갈아타기를 세지 않습니다.** 4편까지 늘어나면서 5,096km 로 가장 짧아집니다. 짧은 구간을 여러 번 이어 붙인 길이라, 지도에서는 예쁘지만 실제로 타기에는 가장 나쁩니다.

수방(SZB)과 쿠알라룸푸르(KUL)는 같은 도시의 다른 공항입니다. 데이터가 **공항 단위**라 생기는 일이니 결과에 함께 밝혀야 합니다.

### 정의가 답을 정한다

**`hours` 는 데이터에 있던 값이 아니라 우리가 정한 값**입니다. 준비 셀에서 이렇게 새겼습니다.

```
hours = km / 800 + 1
```

순항 속도 시속 800km 에 이착륙과 갈아타기 1시간을 더한 값이라 **총 시간 = 총 거리 / 800 + (편 수 × 1시간)** 입니다. 편 수가 곧 벌점입니다.

이 1시간을 바꾸면 답이 바뀝니다. **2.2시간만 넘어도 순서가 뒤집힙니다.** 2시간으로 잡으면 시간 최소 길이 12.47시간, 편 수 최소 길이 12.65시간이라 **덜 갈아타는 쪽이 이깁니다.** 갈아타기를 비싸게 볼수록 편 수가 적은 길이 유리해지기 때문입니다.

**보고서에 적어야 하는 것은 답이 아니라 정의입니다.** "가장 빠른 길"에는 "환승을 몇 시간으로 봤는가"가 숨어 있고, 그 숫자를 정한 사람이 답을 정한 셈입니다.

In [ ]:
# 직항이 있는데도 갈아타는 편이 거리로는 짧은 경우
DEST = 'PNH'   # 프놈펜
# 직항 한 편의 거리. 왕복이면 관계가 둘이라 min 으로 하나만 남긴다
direct = run_cypher('''
    MATCH (a:Airport { iata: $a })-[r:ROUTE]-(b:Airport { iata: $b })
    RETURN min(r.km) AS km''',
    a='ICN', b=DEST)[0]['km']
# km 를 비용으로 준 최단 거리 경로. 갈아타더라도 거리 합이 가장 작은 길이 나온다
best = run_cypher('''
    MATCH (a:Airport { iata: $a }), (b:Airport { iata: $b })
    CALL gds.shortestPath.dijkstra.stream('air',
         { sourceNode: a, targetNode: b, relationshipWeightProperty: 'km' })
    YIELD totalCost, nodeIds
    RETURN toInteger(totalCost) AS km,
           [n IN nodeIds | gds.util.asNode(n).iata] AS route''',
    a='ICN', b=DEST)[0]
print('직항 거리:', direct, 'km')

In [ ]:
print('거리 최소 경로:', best['route'], best['km'], 'km')

> 프놈펜(PNH)에는 3,598km 직항이 있습니다. 그런데 거리 최소 길은 광저우(CAN)를 거쳐 3,597km 입니다. **1km 를 아끼려고 한 번 갈아탑니다.** 시간으로 재면 직항이 이깁니다.

틀린 것은 알고리즘이 아니라 **무엇을 최소화할지 정한 사람**입니다.

드문 일이 아닙니다. 인천(ICN)에서 닿는 766곳을 전부 재 보면, 거리 최소가 편 수 최소보다 더 갈아타는 곳이 **27%**, 시간 최소가 더 갈아타는 곳이 **8%**, 거리 최소와 시간 최소가 다른 길인 곳이 **23%** 입니다. 네 곳 중 한 곳꼴로 어느 잣대를 쓰느냐에 따라 답이 달라집니다.

### 🖐️ 함께 따라하기: 전철에서 정거장 수 최소와 거리 최소

같은 질문을 수도권 전철망에 던져 봅니다. 역이 659개, 구간이 778개(쌍 757)이고, 관계에 `km` (두 역 좌표 사이 직선거리)가 있습니다.

`'subway'` 투영에서 **강남에서 홍대입구까지**를 **두 번** 재세요. 한 번은 가중치 없이(정거장 수 최소), 한 번은 `relationshipWeightProperty: 'km'` 로(거리 최소). 각각 **정거장 수**·**총 km**·**지나는 역**을 출력합니다.

**확인 기준**: 정거장 수 최소는 **10정거장 13.48km**, 거리 최소는 **11정거장 13.13km** 입니다. **한 정거장을 더 가면서 거리는 짧아집니다.** 두 길 모두 같은 값의 경로가 하나뿐이라, 지나는 역까지 그대로 나와야 맞습니다.

> 본선과 떨어진 조각이 하나 있습니다(659개 역이 656개와 3개로 나뉩니다). 작은 쪽 공항 셔틀 구간을 출발지로 잡으면 결과가 비어 있습니다. **길이 없으면 Dijkstra 는 빈 결과를 돌려줍니다.** 오류가 아닙니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) [('정거장 수 최소', None), ('거리 최소', 'km')] 를 for 로 돌린다
# 2) 가중치가 있으면 relationshipWeightProperty 를 설정에 넣는다(2-3 셀의 weight_line 과 같은 방식)
# 3) MATCH 로 :Station 두 개를 name 으로 찾아 dijkstra 를 부르고
#    size(nodeIds) - 1 로 정거장 수, asNode(n).name 으로 역 이름 목록을 받는다
# 4) 그 길의 총 km 는 구간마다 min(r.km) 을 더해 따로 잰다(2-3 의 totals 셀과 같다)
# 5) 구간마다 collect(DISTINCT r.line) 로 '탈 수 있는 노선'도 받아 출력한다(환승을 세는 근거)
# 6) 기준, 정거장 수, km, 경로를 표로 출력한다

> 두 길이 갈리는 곳은 출발 직후입니다. 정거장 수 최소는 교대역으로 바로 넘어가고, 거리 최소는 신논현역을 거칩니다. 한 정거장을 더 가지만 총 거리는 13.13km 로 13.48km 보다 짧습니다.

실제로 표를 끊는 사람에게는 세 번째 잣대가 있습니다. **환승 횟수**입니다. 위 셀이 함께 찍은 **구간별 노선**을 앞에서부터 이어 보며 같은 노선을 최대한 오래 타도록 고르면, 정거장 수 최소 길은 5번, 거리 최소 길은 4번입니다. 정거장이 하나 적은 길이 환승은 더 많습니다. **세 잣대가 다른 답을 가리킬 때 무엇을 고를지는 데이터가 아니라 사람이 정합니다.**

> 여기 `km` 는 **두 역 좌표 사이의 직선거리**이고 실제 선로 길이가 아닙니다. 곡선 구간이 많은 노선은 짧게 잡힙니다.

### ✅ 바로 확인 퀴즈

**1.** `gds.shortestPath.dijkstra` 에 가중치를 주지 않으면 `totalCost` 는 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

**건너간 관계의 수**입니다. 관계 하나가 비용 1 이라 노선망에서는 편 수, 전철에서는 정거장 수입니다.

</details>

**2.** 같은 두 공항인데 시간 최소 경로가 편 수 최소 경로보다 **한 번 더** 갈아탑니다. 왜일까요?

<details><summary>정답 보기</summary>

편 수가 적은 길이 **크게 돌아가기** 때문입니다. 갈아타기 1시간보다 돌아가며 잃는 시간이 크면 한 번 더 갈아타는 길이 이깁니다.

</details>

**3.** 같은 쿼리를 두 번 돌렸더니 경로에 나온 경유지가 달랐습니다. 오류인가요?

<details><summary>정답 보기</summary>

아닙니다. **같은 값의 최적 경로가 여럿**이면 어느 것을 돌려줄지 정해져 있지 않습니다. 가중치 없이 잰 최단 경로에서 특히 잦습니다. 총비용(편 수)은 항상 같으니, 보고할 때는 경로보다 **거리나 편 수**를 근거로 쓰는 편이 안전합니다.

</details>

---
# 3. 셋을 겹쳐 대체 공항 찾기

## 3-1. 네 조건을 한 번에 걸어 후보 뽑기

### 왜 필요할까요?
"이 공항이 태풍으로 닫혔습니다. 어디로 돌릴까요?" 이 질문에는 한 가지 잣대로 답할 수 없습니다. **오늘 배운 도구 세 가지로 네 가지 조건을 겁니다.**

- **같은 커뮤니티**에 있다: 노선이 촘촘히 오가는 같은 덩어리 안이다 (교안_01 의 커뮤니티).
- **국가·지역이 다르다**: 같은 나라 안에서 돌리는 것은 이미 하고 있다.
- **노선망의 자리가 닮았다**: 상대 공항 목록이 겹친다 (1절의 유사도).
- **두 편 안에 닿는다**: 승객을 실제로 보낼 수 있는 거리다 (2절의 최단 경로).

네 조건을 다 만족하는 짝이 **지도만 봐서는 안 보이는 대체 공항**입니다.

In [ ]:
# 커뮤니티를 노드에 저장해 둔다. 아래 분석이 이 번호를 계속 쓰므로 randomSeed 와 concurrency 를 고정한다
run_cypher('''
    CALL gds.leiden.write('air', { writeProperty: 'community',
                                   randomSeed: 42, concurrency: 1 })
    YIELD communityCount RETURN communityCount''')
print(run_cypher('MATCH (a:Airport) RETURN count(DISTINCT a.community) AS communities')[0])

In [ ]:
# 네 조건을 한 번에 건다. topK 를 넉넉히 10 으로 둬야 걸러 내고도 후보가 남는다
alt_rows = run_cypher('''
    CALL gds.nodeSimilarity.stream('air', { topK: 10, degreeCutoff: 10 })
    YIELD node1, node2, similarity
    WITH gds.util.asNode(node1) AS a, gds.util.asNode(node2) AS b, similarity
    // 같은 쌍이 양쪽으로 나올 수도 한쪽만 나올 수도 있다. 코드 순서로 맞춘 뒤 DISTINCT 로 한 줄만 남긴다
    WITH CASE WHEN a.iata < b.iata THEN a ELSE b END AS x,
         CASE WHEN a.iata < b.iata THEN b ELSE a END AS y, similarity
    WHERE x.community = y.community
      AND x.country <> y.country
      AND EXISTS { MATCH (x)-[:ROUTE*1..2]-(y) }
    RETURN DISTINCT x.iata AS airport_a, x.city AS city_a, x.country AS country_a,
           y.iata AS airport_b, y.city AS city_b, y.country AS country_b,
           round(similarity, 4) AS similarity
    ORDER BY similarity DESC, airport_a LIMIT 10''')
display(pd.DataFrame(alt_rows))

> 이 명단이 오늘의 결론입니다. 1위는 **쿠알라룸푸르(KUL)와 싱가포르(SIN)(유사도 0.5868)** 입니다. 나라가 다른데 같은 커뮤니티에 있고, 상대 공항 목록이 절반 넘게 겹치며, 서로 직항이 있습니다. **노선망에서 같은 자리**에 서 있는 것입니다.

> 커뮤니티는 `randomSeed` 42 로 고정해도 **적재할 때마다** 7개에서 8개 사이에서 달라집니다(교안_01 5-2). 그래도 이 표의 위쪽은 잘 흔들리지 않습니다. **유사도와 노선 거리는 흔들리지 않는 값**이라, 세 조건 가운데 둘이 명단을 붙잡아 주기 때문입니다.

실무에서 이 표를 받으면 다음 질문이 나옵니다. "이 공항에 우리 노선을 받아 줄 슬롯이 있나?" 데이터는 여기까지고, 그다음은 사람에게 물어야 합니다. **분석의 산출물은 답이 아니라 좋은 질문일 때가 많습니다.**

<img src="images/교안/두_국가_사이_노선.png" width="860">

### 한 커뮤니티에 함께 들어간 두 나라 들여다보기

커뮤니티 하나를 골라 안을 봅시다. `randomSeed` 를 주지 않고 160번 돌려도 **일본과 대한민국은 160번 모두 한 커뮤니티**입니다. 실행마다 흔들리는 알고리즘인데도 이 둘만은 늘 붙어 다닙니다.

두 나라의 공항은 61곳과 15곳으로, **사이**만 보기에 좋은 규모입니다.

In [ ]:
# 두 나라의 규모와, 두 나라 '사이'에 놓인 노선이 얼마나 되는지 센다
LEFT, RIGHT = 'Japan', 'South Korea'   # randomSeed 없이 여러 번 돌려도 늘 한 커뮤니티던 두 나라
facts = run_cypher('''
    MATCH (a:Airport { country: $left })
    WITH count(a) AS left_airports
    MATCH (b:Airport { country: $right })
    WITH left_airports, count(b) AS right_airports
    // 왕복 노선은 관계가 둘이라, 공항 쌍으로 셀 때는 DISTINCT 가 필요하다
    MATCH (x:Airport { country: $left })-[r:ROUTE]-(y:Airport { country: $right })
    RETURN left_airports, right_airports,
           count(DISTINCT [x.iata, y.iata]) AS pairs, count(r) AS rels''',
    left=LEFT, right=RIGHT)
print(facts[0])   # pairs 는 두 나라를 잇는 공항 쌍 수, rels 는 그 관계 수(왕복이면 둘)

In [ ]:
# 그 나라 공항 가운데 몇 곳이나 상대 나라와 노선이 있는지도 본다
touched = run_cypher('''
    MATCH (a:Airport { country: $left })
    WHERE EXISTS { MATCH (a)-[:ROUTE]-(:Airport { country: $right }) }
    RETURN count(a) AS touched''', left=LEFT, right=RIGHT)
print(f"{LEFT} 에서 {RIGHT} 와 노선이 있는 공항:", touched[0]['touched'], '곳')

> 일본은 공항이 61곳, 대한민국은 15곳인데 **두 나라를 잇는 공항 쌍이 38개**입니다(방향별 관계로는 76개, 전부 왕복이라는 뜻입니다). 일본 쪽 25곳, 대한민국 쪽 4곳이 국경 너머와 이어져 있습니다.

다만 그 왕래가 **고르게 퍼져 있는지 몇 곳에 몰려 있는지**는 숫자로 알 수 없습니다. 처방이 달라지는 차이라 그림으로 확인합니다. 두 나라를 좌우 두 줄로 세우고 **두 나라를 잇는 노선만** 그립니다.

In [ ]:
import platform

import matplotlib.pyplot as plt

# 한글 폰트: 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == "Windows":
    KOREAN_FONT = "Malgun Gothic"
elif platform.system() == "Darwin":          # macOS
    KOREAN_FONT = "AppleGothic"
else:                                        # Linux (Colab 등)
    KOREAN_FONT = "NanumGothic"

plt.rcParams["font.family"] = KOREAN_FONT    # 이후 모든 그림에 이 폰트가 적용된다
plt.rcParams["axes.unicode_minus"] = False   # 마이너스(-) 부호 깨짐 방지
import networkx as nx

# 1) 왼쪽 줄에 세울 공항: LEFT 나라 공항 중 RIGHT 나라와 노선이 있는 곳
left_ids = [r['iata'] for r in run_cypher('''
    MATCH (a:Airport { country: $left })
    WHERE EXISTS { MATCH (a)-[:ROUTE]-(:Airport { country: $right }) }
    RETURN a.iata AS iata ORDER BY iata''',
    left=LEFT, right=RIGHT)]
# 2) 오른쪽 줄에 세울 공항: RIGHT 나라 공항 중 LEFT 나라와 노선이 있는 곳
right_ids = [r['iata'] for r in run_cypher('''
    MATCH (b:Airport { country: $right })
    WHERE EXISTS { MATCH (b)-[:ROUTE]-(:Airport { country: $left }) }
    RETURN b.iata AS iata ORDER BY iata''',
    left=LEFT, right=RIGHT)]
# 3) 두 나라를 잇는 선. 왕복 노선은 관계가 둘이라 DISTINCT 로 한 번만 받는다
links = run_cypher('''
    MATCH (a:Airport { country: $left })-[:ROUTE]-(b:Airport { country: $right })
    RETURN DISTINCT a.iata AS source, b.iata AS target''',
    left=LEFT, right=RIGHT)
print('왼쪽', len(left_ids), '곳 · 오른쪽', len(right_ids), '곳 · 잇는 선', len(links), '개')

# links 한 줄이 '왼쪽 한 곳 + 오른쪽 한 곳' 이라, source 를 세면 왼쪽 공항의 선 수가 된다
from collections import Counter
left_count = Counter(r['source'] for r in links)
right_count = Counter(r['target'] for r in links)
# most_common(1) 은 [(공항, 선 수)] 한 칸짜리 리스트라 [0] 으로 그 칸을 꺼낸다
left_top, right_top = left_count.most_common(1)[0], right_count.most_common(1)[0]
# 값이 1 인 공항 = 상대 나라에서 딱 한 곳하고만 이어진 공항
left_single = sum(1 for v in left_count.values() if v == 1)
right_single = sum(1 for v in right_count.values() if v == 1)
print(f'선이 가장 많은 공항: 왼쪽 {left_top[0]} {left_top[1]}쌍 · 오른쪽 {right_top[0]} {right_top[1]}쌍')
print(f'선이 한 가닥뿐인 공항: 왼쪽 {left_single}곳 · 오른쪽 {right_single}곳')

In [ ]:
across = nx.Graph()
across.add_nodes_from(left_ids)
across.add_nodes_from(right_ids)
across.add_edges_from([(r['source'], r['target']) for r in links])

# 자리를 직접 정한다. 좌우 두 줄로 세워야 '나라 사이에 몇 줄이 오가는가'가 한눈에 들어온다
pos = {}
for i, node in enumerate(left_ids):
    pos[node] = (0.0, -i / (len(left_ids) - 1))
for i, node in enumerate(right_ids):
    pos[node] = (1.0, -i / (len(right_ids) - 1))

fig, ax = plt.subplots(figsize=(8, 9))
# 선이 겹치므로 alpha 를 낮춰 겹친 정도가 진하기로 보이게 한다
nx.draw_networkx_edges(across, pos, ax=ax, edge_color='#C44E52', alpha=0.35, width=0.9)
nx.draw_networkx_nodes(across, pos, ax=ax, nodelist=left_ids, node_color='#4C72B0',
                       node_size=380, edgecolors='white', linewidths=1.0)
# 두 나라를 색으로 갈라야 어느 쪽 공항인지 보인다
nx.draw_networkx_nodes(across, pos, ax=ax, nodelist=right_ids, node_color='#DD8452',
                       node_size=380, edgecolors='white', linewidths=1.0)
nx.draw_networkx_labels(across, pos, ax=ax, font_size=7, font_color='white')
ax.text(0.0, 0.08, f'{LEFT} ({len(left_ids)}곳)', ha='center',
        fontfamily=KOREAN_FONT, fontsize=12, color='#4C72B0')
ax.text(1.0, 0.08, f'{RIGHT} ({len(right_ids)}곳)', ha='center',
        fontfamily=KOREAN_FONT, fontsize=12, color='#DD8452')
ax.set_title(f'두 나라 사이에 놓인 노선 {len(links)}쌍', fontfamily=KOREAN_FONT)
ax.set_xlim(-0.25, 1.25)
ax.axis('off')
plt.show()

> 두 가지가 함께 보입니다. **닿는 범위는 넓습니다.** 일본의 25곳, 대한민국의 4곳이 국경 너머와 한 가닥이라도 이어져 있습니다. **그런데 선이 한쪽으로 몰려 있습니다.** 인천(ICN) 한 곳이 38쌍 가운데 25쌍, 전체의 66% 를 차지합니다. 건너편에서 가장 많은 오사카 간사이(KIX)도 4쌍뿐이라, 몰림은 대한민국 쪽에서만 일어납니다. 반대로 **선이 한 가닥뿐인 공항**은 위 셀 출력에서 왼쪽, 곧 일본 쪽이 훨씬 많습니다.

> **두 나라는 넓게 닿아 있지만, 그 왕래를 떠받치는 것은 대한민국 쪽 관문 공항 한 곳**입니다. 커뮤니티 탐지가 두 나라를 한 커뮤니티로 본 것은 **정확한 관찰**이었고, 그 커뮤니티를 지탱하는 공항이 어디인지까지 그림이 알려 줍니다.

노선 기획팀에 보여 주면 질문이 두 개 나옵니다. "인천(ICN)이 닫히면 두 나라 사이가 어떻게 되나요?" 와 "한 가닥뿐인 공항들을 이어 줄 노선을 열 만한가요?" **분석이 의사결정에 닿는 지점**입니다.

### 🖐️ 함께 따라하기: 경로가 지나는 부서 세기

마지막으로 메일망으로 돌아갑니다. `'team'` 투영에서 **377번에서 75번까지**의 최단 경로를 구하고, 그 경로가 **서로 다른 부서를 몇 개나 지나는지** 세어 출력하세요. 파이썬의 `set` 을 쓰면 한 줄입니다.

**확인 기준**: **4다리**입니다. 이 두 사람 사이에는 같은 길이의 최단 경로가 **22개**나 있어서, 지나는 사람도 그 부서도 **실행마다 달라집니다.** 서로 다른 부서 수는 대개 2개로 나오지만 더 나올 수도 있습니다. 수를 맞히는 것이 목적이 아닙니다.

**무엇이 나와도 변하지 않는 것**: 다리 수와 양 끝 두 사람의 부서입니다. 경로 자체가 아니라 **양 끝과 길이가 불변**이라는 것이 최단 경로를 읽는 요령입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 :Member 두 명을 employee_id 로 찾는다(지문의 두 번호)
# 2) gds.shortestPath.dijkstra.stream('team', {sourceNode: a, targetNode: b}) 을 호출한다
# 3) totalCost 와 nodeIds 를 받아 employee_id, dept 목록을 만든다
# 4) 부서 목록을 set 으로 묶어 서로 다른 부서가 몇 개인지 센다

### ✅ 바로 확인 퀴즈

**1.** "같은 커뮤니티고 국가가 다르고 유사도가 높고 두 편 안에 닿는다"는 네 조건을 다 걸어 뽑은 짝은 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

**노선망에서 같은 자리에 서 있으면서 국경만 다른 공항 짝**입니다. 한쪽이 막혔을 때 승객과 노선을 옮겨 받을 수 있는 후보입니다.

</details>

**2.** 두 나라를 잇는 선이 **양쪽에 고르게** 퍼져 있었다면, 같은 그림을 어떻게 다르게 읽어야 할까요?

<details><summary>정답 보기</summary>

관문 공항 한두 곳에 기대지 않고 **여러 갈래로 이어져 있다**는 뜻입니다. 공항 하나가 닫혀도 왕래가 남습니다. 지금 그림은 그 반대라, 인천(ICN)이 닫히면 두 나라를 잇는 38쌍 가운데 25쌍(66%)이 사라집니다.

</details>

**3.** 최단 경로가 지나는 부서 수가 **1** 로 나왔습니다. 무슨 뜻인가요?

<details><summary>정답 보기</summary>

출발부터 도착까지 **한 부서 안에서만** 이어졌다는 뜻입니다. 두 사람이 같은 부서입니다.

</details>

---
## 🚀 응용 클론코딩: 대체 공항 추천 함수

"이 공항이 다음 주에 닫힙니다. 어디로 돌릴 수 있을까요?"에 답하는 함수를 만드세요.

`find_alternative(iata)` 는 그 공항의 **대체 후보**를 리스트로 돌려줍니다. 각 원소는 `airport`, `city`, `country`, `similarity`, `direct` 다섯 키를 가진 사전입니다. `direct` 는 그 후보와 **직항으로 이어져 있으면** `True` 입니다.

조건은 3-1 과 같습니다. **같은 커뮤니티 · 국가가 다름 · 두 편 안에 닿음**이고, 유사도는 `degreeCutoff: 10`, `topK: 10` 으로 구합니다. 물어본 공항이 결과의 왼쪽에 올 수도 오른쪽에 올 수도 있으니 **양쪽을 다 받아** 방향을 맞춰야 합니다.

만든 뒤 **쿠알라룸푸르(KUL)** 에 적용해 출력하세요.

**확인 기준**: 1위는 **싱가포르(SIN)**, 유사도 **0.5868** 입니다. 3-1 표의 1위 줄과 같은 쌍입니다.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) def find_alternative(iata): 로 함수를 만든다
# 2) gds.nodeSimilarity.stream('air', {topK: 10, degreeCutoff: 10}) 을 호출하고
#    WHERE a.iata = $who OR b.iata = $who 로 물어본 공항이 낀 행만 남긴다
# 3) CASE WHEN 으로 '물어본 공항'과 '상대'를 각각 me, other 로 정리한다
# 4) me 와 other 가 같은 커뮤니티고 국가가 다르고 EXISTS { MATCH (me)-[:ROUTE*1..2]-(other) } 인 것만 남긴다
# 5) 상대의 iata, city, country, 유사도와, 직항 여부를 유사도 내림차순으로 돌려준다
# 6) 결과를 DataFrame 으로 출력한다

---
## 이번 강의 정리

| 한 일 | 쓴 것 | 기억할 것 |
|---|---|---|
| 닮은 공항 찾기 | `gds.nodeSimilarity.stream` | 교집합 나누기 합집합. 값이 아니라 **정의**를 기억한다 |
| 함정 피하기 | `degreeCutoff` | 상대가 한두 곳이면 유사도가 쉽게 1.0 이 된다 |
| 상위만 받기 | `topK` | 공항마다 상위 K 곳. 서로의 상위에 들면 (A,B)와 (B,A)가 둘 다 나온다 |
| 몇 편인가 | `gds.shortestPath.dijkstra` | 가중치를 안 주면 `totalCost` 가 곧 편 수. 동점이 흔하다 |
| 전체까지의 거리 | `gds.allShortestPaths.dijkstra` | 허브의 이점은 도달 범위가 아니라 **직항 수**에서 나온다 |
| 거리·시간 최소 | `relationshipWeightProperty` | 무엇을 최소화할지 정하는 것이 답을 정한다 |
| 셋 겹치기 | 커뮤니티 + 국가 + 유사도 + 편 수 | 흔들리는 도구와 흔들리지 않는 도구를 섞는다 |

**오늘의 결론.** 지도는 공항을 위치로 늘어놓지만, 노선 데이터는 **어느 공항이 어느 공항의 자리를 대신할 수 있는지**를 보여 줍니다. 그리고 그 답은 우리가 **무엇을 세고 무엇을 최소화하기로 했는지**에 매여 있습니다.

## 자료 출처

남의 데이터를 쓸 때는 **라이선스와 갱신 시점**을 함께 적어야 합니다. 갱신이 멈춘 자료로 낸 결론은 그때의 것입니다.

- **아시아 항공 노선망**(시연): OpenFlights `airports.dat`·`routes.dat`(ODbL). 노선은 **2014년 6월에 갱신이 멈춘 자료**라 시간표가 아니라 노선 유무만 담겨 있습니다.
- **다섯 부서 메일망**(1절·3절 따라하기): SNAP `email-Eu-core`(라이선스 표기 없음, 학습용). 유럽 연구기관의 사내 메일을 익명화한 공개 데이터입니다.
- **수도권 전철망**(2절 따라하기): OpenStreetMap contributors(ODbL). 30일차와 같은 파일이고 `km` 는 좌표 사이 직선거리입니다.

## ⏭️ 예고: 35일차

지금까지는 **그래프의 구조**만 봤습니다. 35일차에서는 노드에 **글**을 붙입니다. 문장의 뜻을 숫자로 바꾼 임베딩을 심고, 구조 검색과 의미 검색을 함께 씁니다. 오늘의 **커뮤니티**과 앞서 배운 **PageRank** 가 검색 범위를 좁히고 순위를 매기는 데 쓰입니다.

- 35일차: 벡터 검색과 GraphRAG